# 01b · A fresher, richer data source

The original files stop in **March 2024**. To refresh them we tried ufcstats.com directly, but every page now returns a
*"Checking your browser…"* JavaScript challenge, which is the site saying no to automated scrapers. We respect that rather than work around it.

Instead we use the public GitHub repo **[Greco1899/scrape_ufc_stats](https://github.com/Greco1899/scrape_ufc_stats)** (GPL‑3.0).
It republishes ufcstats data, updated regularly, and it has more detail than our old files:

| | Old files (2024) | Greco1899 snapshot |
|---|---|---|
| Fights | ~7,500, to 2024‑03 | ~8,950, to 2026‑10 |
| Stats | whole-fight totals | **round by round** |
| Strike detail | significant strikes only | **head / body / leg, distance / clinch / ground** |
| Age | approximate (2024 age shifted back) | **exact date of birth** |
| Judges | none | **every judge's scorecard** |

`src/ufc/ingest.py` downloads a dated snapshot (`ingest.download()`) and builds the same tables as before, plus round and judge tables.

In [1]:
import os, sys
from pathlib import Path
# Run from the project root no matter where Jupyter was started.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)

In [2]:
from ufc import ingest
snap = ingest.latest_snapshot()
print("snapshot:", snap)
for f in sorted(snap.glob("*.csv")):
    print(f"{f.name:28s} {sum(1 for _ in open(f, encoding='utf-8')) - 1:>7,} rows")

snapshot: data/raw/greco/2026-10-19
ufc_event_details.csv            793 rows
ufc_fight_details.csv          8,975 rows
ufc_fight_results.csv          8,975 rows
ufc_fight_stats.csv           42,162 rows
ufc_fighter_details.csv        4,628 rows
ufc_fighter_tott.csv           4,629 rows


In [3]:
raw = pd.read_csv(snap / "ufc_fight_stats.csv")
raw.head(3).T

,0,1,2
EVENT,UFC Fight Night: Buckley vs. Malott,UFC Fight Night: Buckley vs. Malott,UFC Fight Night: Buckley vs. Malott
BOUT,Joaquin Buckley vs. Mike Malott,Joaquin Buckley vs. Mike Malott,Joaquin Buckley vs. Mike Malott
ROUND,Round 1,Round 2,Round 1
FIGHTER,Joaquin Buckley,Joaquin Buckley,Mike Malott
KD,0.0,0.0,0.0
SIG.STR.,9 of 37,5 of 16,15 of 25
SIG.STR. %,24%,31%,60%
TOTAL STR.,9 of 37,11 of 22,15 of 25
TD,0 of 0,0 of 0,0 of 0
TD %,---,---,---


Each stat is a string like `"9 of 37"` (landed of attempted), and control time is `"1:23"`. The parsers in `ingest.py` turn these into numbers.

## Gotchas found in this source
1. **Duplicate events.** One event appears twice under two names ("Noche UFC: Lopes vs. Silva" and "UFC Fight Night: Lopes vs. Silva"). The fight **URL** is the real ID, so we deduplicate on it.
2. **Spelling variants.** A bout lists "YiSak Lee" while the profile says "Yi Sak Lee". We match after removing spaces, hyphens and suffixes (Jr., III), plus a short manual list for nicknames (Cam → Cameron). Plain fuzzy matching was tried and rejected: it wrongly matched "Muhammad Saidov" to a different fighter, "Muhammad Said".
3. **Different people with one name.** There are two Bruno Silvas and two Jean Silvas. Each fight is assigned to the profile whose listed weight fits the division and whose birthdate gives a plausible age.
4. **Scorecards are written loser‑first.** "28 - 29" means the loser got 28 from that judge. A split decision reads "28 - 29. 29 - 28. 28 - 29.", so the dissenting judge's card is the flipped one.
5. **Same red‑corner artifact before 2010‑03‑21** as the old files, so the same cutoff applies.

In [4]:
fights, ff, rounds, judges, report = ingest.run()
print(fights.shape, ff.shape, rounds.shape, judges.shape)
for k, v in report.items():
    print(f"{k}: {v}")

(8947, 80) (17894, 62) (41996, 27) (12100, 6)
undated_events: ['UFC - Road to UFC 4.6']
undated_fights: 2
name_aliases: {'Zach Reese': 'Zachary Reese', 'Waldo Cortes Acosta': 'Waldo Cortes-Acosta', 'Shem Rock': 'Shaqueme Rock', 'Sean King III': 'Sean King', 'Cam Nelson': 'Cameron Nelson', 'Ilimbek Akylbek': 'Ilimbek Akylbek Uulu', 'Rafael Cerquiera': 'Rafael Cerqueira', 'Michael Aswell Jr.': 'Michael Aswell', 'YiSak Lee': 'Yi Sak Lee', 'Levi Rodrigues Jr.': 'Levi Rodrigues', 'Kai Kamaka III': 'Kai Kamaka'}
split_names: ['Bruno Silva (1989)', 'Bruno Silva (1990)', 'Jean Silva (1977)', 'Jean Silva (1996)']


## Cross-check against the old dataset
Before switching sources, every fight the two datasets share should agree.

In [5]:
from ufc import cleaning
from ufc.cleaning import norm_name
old = cleaning.build_fights(*cleaning.load_raw("data/raw"))
strip = lambda n: norm_name(str(n).split(" (")[0])
key = lambda df: df.date.dt.strftime("%Y-%m-%d") + "|" + df.r_fighter.map(strip) + "|" + df.b_fighter.map(strip)
m = old.assign(k=key(old)).merge(fights.assign(k=key(fights)), on="k", suffixes=("_old", "_new"))
both = m.r_ctrl_sec_old.notna() & m.r_ctrl_sec_new.notna()
checks = pd.Series({
    "old fights found in new": len(m) / len(old),
    "same winner": (m.winner_old == m.winner_new).mean(),
    "same method group": (m.method_group_old == m.method_group_new).mean(),
    "same sig strikes (red)": (m.r_sig_str_old == m.r_sig_str_new).mean(),
    "same takedowns (red)": (m.r_td_old == m.r_td_new).mean(),
    "control time within 5s": ((m.r_ctrl_sec_old - m.r_ctrl_sec_new).abs() <= 5)[both].mean(),
})
checks.map("{:.2%}".format)

old fights found in new     99.21%
same winner                100.00%
same method group          100.00%
same sig strikes (red)      99.72%
same takedowns (red)        99.72%
control time within 5s     100.00%
dtype: object

In [6]:
old[~key(old).isin(set(m.k))][["date", "r_fighter", "b_fighter"]].head(6)

,date,r_fighter,b_fighter
2168,2013-02-02,Jacob Volkmann,Bobby Green
2450,2013-11-06,Bobby Green,James Krause
2504,2013-12-14,Bobby Green,Pat Healy
2808,2014-07-26,Josh Thomson,Bobby Green
2969,2014-11-22,Bobby Green,Edson Barboza
2984,2014-12-12,Tecia Torres,Angela Magana


The few unmatched fights are mostly **name changes** (Bobby Green is now "King Green", Tecia Torres is now "Tecia Pennington"). The new source uses current names. Control time differs by a few seconds because the new source adds up rounded per-round values, and early fights where control wasn't tracked are now blank instead of 0.

### Judges sanity check
Unanimous decisions should have every judge picking the winner, and split decisions exactly 2 of 3.

In [7]:
j = judges.merge(fights[["fight_id", "winner", "method"]], on="fight_id")
j = j[j.winner.isin(["Red", "Blue"])]
picked = np.where(j.winner == "Red", j.r_score > j.b_score, j.b_score > j.r_score)
j.assign(picked_winner=picked).groupby("method").picked_winner.mean().round(3)

method
Decision - Majority     0.672
Decision - Split        0.666
Decision - Unanimous    1.000
Name: picked_winner, dtype: float64

### Rounds add up
Summing the round table gives back the fight totals exactly.

In [8]:
tot = rounds.groupby(["fight_id", "corner"]).sig_str.sum().unstack()
f = fights.set_index("fight_id").loc[tot.index]
print("red totals match:", (tot.Red == f.r_sig_str).mean(), "| blue totals match:", (tot.Blue == f.b_sig_str).mean())
rounds.head()

red totals match: 1.0 | blue totals match: 1.0


,fight_url_id,corner,round,kd,sig_str,sig_str_att,str,str_att,td,td_att,sub_att,rev,ctrl_sec,head,head_att,body,body_att,leg,leg_att,distance,distance_att,clinch,clinch_att,ground,ground_att,fight_id,fighter
0,4acab67848e78327,Blue,1,0.0,0.0,4.0,1.0,5.0,0.0,0.0,0.0,0.0,NaN,0.0,2.0,0.0,0.0,0.0,2.0,0.0,3.0,0.0,1.0,0.0,0.0,0,Sean Daugherty
1,4acab67848e78327,Red,1,0.0,1.0,1.0,2.0,2.0,1.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0,Scott Morris
2,4b9ae533ccb3fcdf,Blue,1,0.0,1.0,1.0,2.0,2.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1,Ray Wizard
3,4b9ae533ccb3fcdf,Red,1,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,NaN,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1,Patrick Smith
4,c53325be8ce33b8b,Blue,1,0.0,0.0,2.0,14.0,23.0,1.0,1.0,0.0,2.0,NaN,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,2,Scott Baker


From here on, every notebook uses this source. → `02_cleaning.ipynb`